In [1]:
%%time
%%capture
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install accelerate

CPU times: user 17 ms, sys: 9.57 ms, total: 26.6 ms
Wall time: 14.2 s


In [2]:
%%time
import shutil

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

CPU times: user 4.15 s, sys: 471 ms, total: 4.62 s
Wall time: 3.42 s


In [3]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [4]:
%%time
tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")
base_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/mdeberta-v3-base", num_labels=2
)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

/opt/conda/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 5.61 s, sys: 3.8 s, total: 9.41 s
Wall time: 6.3 s


In [5]:
df = pd.read_csv("pretrain.csv")
print("Full dataset shape:", df.shape)
print("Overall label distribution (%):")
print(df["labels"].value_counts(normalize=True) * 100)
print(df.dtypes)

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Full dataset shape: (118802, 2)
Overall label distribution (%):
labels
0    54.420801
1    45.579199
Name: proportion, dtype: float64
text      object
labels     int64
dtype: object


In [6]:
df = df.dropna(subset=["text", "labels"])
df["text"] = df["text"].astype(str)

In [7]:
print("Missing values in text column:", df["text"].isna().sum())
print("Missing values in labels column:", df["labels"].isna().sum())
print("Any NaN in labels?", df["labels"].isna().sum())
print("Full dataset shape:", df.shape)

Missing values in text column: 0
Missing values in labels column: 0
Any NaN in labels? 0
Full dataset shape: (118799, 2)


In [8]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.10,  # 10% goes to temp
    stratify=df["labels"],  # keep class balance
    random_state=42,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,  # split 10% into 5% + 5%
    stratify=temp_df["labels"],
    random_state=42,
    shuffle=True,
)

print("Shapes:")
print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

print("\nLabel distribution (%)")

print("\nTrain:")
print(train_df["labels"].value_counts(normalize=True) * 100)

print("\nValidation:")
print(val_df["labels"].value_counts(normalize=True) * 100)

print("\nTest:")
print(test_df["labels"].value_counts(normalize=True) * 100)

Shapes:
Train: (106919, 2)
Val:   (5940, 2)
Test:  (5940, 2)

Label distribution (%)

Train:
labels
0    54.419701
1    45.580299
Name: proportion, dtype: float64

Validation:
labels
0    54.410774
1    45.589226
Name: proportion, dtype: float64

Test:
labels
0    54.427609
1    45.572391
Name: proportion, dtype: float64


In [9]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [10]:
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]])
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]])
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]])

In [11]:
dataset = DatasetDict(
    {"train": train_dataset, "validation": val_dataset, "test": test_dataset}
)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 106919
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 5940
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 5940
    })
})

In [12]:
%%time


def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=256
    )


# Encode the input data
dataset = dataset.map(tokenize_function, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/106919 [00:00<?, ? examples/s]

Map:   0%|          | 0/5940 [00:00<?, ? examples/s]

Map:   0%|          | 0/5940 [00:00<?, ? examples/s]

CPU times: user 17.1 s, sys: 1.06 s, total: 18.2 s
Wall time: 10.6 s


In [13]:
%%capture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
base_model.to(device)

In [14]:
# # Add this before training to see what's happening
# print("\n" + "="*50)
# print("PRE-TRAINING DIAGNOSTICS")
# print("="*50)

# # Check a batch
# from torch.utils.data import DataLoader
# train_loader = DataLoader(dataset['train'], batch_size=8)
# batch = next(iter(train_loader))

# print(f"\nBatch shapes:")
# print(f"  input_ids: {batch['input_ids'].shape}")
# print(f"  attention_mask: {batch['attention_mask'].shape}")
# print(f"  labels: {batch['labels'].shape}")
# print(f"\nBatch labels: {batch['labels']}")
# print(f"Labels dtype: {batch['labels'].dtype}")

# # Try a forward pass
# base_model.eval()
# with torch.no_grad():
#     batch = {k: v.to(device) for k, v in batch.items()}
#     outputs = base_model(**batch)
#     print(f"\nForward pass successful!")
#     print(f"Loss: {outputs.loss.item()}")
#     print(f"Logits shape: {outputs.logits.shape}")

In [15]:
steps_per_epoch = len(dataset["train"]) // (32 * 2)
# eval_steps = steps_per_epoch // 3
eval_steps = steps_per_epoch

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Eval steps: {eval_steps}")

Steps per epoch: 1670
Eval steps: 1670


In [16]:
# training_args = TrainingArguments(
#     output_dir="./output_results",
#     num_train_epochs=30,
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=32,
#     #     warmup_steps=500,
#     logging_dir="./logs",
#     learning_rate=2e-5,
#     # weight_decay=0.01,
#     optim="adamw_torch",
#     gradient_accumulation_steps=2,
#     load_best_model_at_end=True,  # needed for early stopping
#     eval_strategy="steps",
#     # logging_steps=500,
#     # eval_steps=500,
#     logging_steps=1000,
#     eval_steps=1000,
#     save_steps=5000,
#     report_to="none",
# )
training_args = TrainingArguments(
    output_dir="./output_results",
    num_train_epochs=30,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=2,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    eval_strategy="steps",
    logging_dir="./logs",
    report_to="none",
    fp16=False,
)

metric = evaluate.load("accuracy")


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)


trainer = Trainer(
    # model=model,
    model=base_model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [17]:
%%time
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
1670,0.564500,0.473173,0.769192
3340,0.438100,0.446602,0.790909
5010,0.381400,0.431717,0.804040
6680,0.327200,0.482136,0.799663
8350,0.274600,0.549698,0.796128
10020,0.225500,0.617437,0.796296


CPU times: user 1h 39min 22s, sys: 4min 20s, total: 1h 43min 42s
Wall time: 1h 43min 58s


TrainOutput(global_step=10020, training_loss=0.3685446672572823, metrics={'train_runtime': 6238.7933, 'train_samples_per_second': 514.133, 'train_steps_per_second': 8.035, 'total_flos': 8.434899702409728e+16, 'train_loss': 0.3685446672572823, 'epoch': 5.99640933572711})

In [18]:
%%time
trainer.evaluate()

CPU times: user 19.7 s, sys: 24.2 ms, total: 19.8 s
Wall time: 19.9 s


{'eval_loss': 0.6174374222755432,
 'eval_accuracy': 0.7962962962962963,
 'eval_runtime': 19.8835,
 'eval_samples_per_second': 298.74,
 'eval_steps_per_second': 9.354,
 'epoch': 5.99640933572711}

In [19]:
%%time
# eval now
predictions = trainer.predict(dataset["test"])

# Extract predicted labels and true labels
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Compute metrics using sklearn's classification report
report = classification_report(
    labels, preds, target_names=["Not Polar (0)", "Polar (1)"], digits=4
)

print(report)

accuracy = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")

               precision    recall  f1-score   support

Not Polar (0)     0.8232    0.7980    0.8104      3233
    Polar (1)     0.7673    0.7953    0.7811      2707

     accuracy                         0.7968      5940
    macro avg     0.7953    0.7967    0.7957      5940
 weighted avg     0.7977    0.7968    0.7970      5940

Accuracy: 0.7968
Macro F1 Score: 0.7957
CPU times: user 19.8 s, sys: 35.5 ms, total: 19.8 s
Wall time: 19.9 s


In [ ]:
# Save the trained model and tokenizer
output_path = "./pretrain_mdeberta"
# Save model
trainer.save_model(output_path)
# Save tokenizer
tokenizer.save_pretrained(output_path)
print(f"Model and tokenizer saved to {output_path}")

# Zip the folder
zip_filename = "pretrain_mdeberta"
shutil.make_archive(zip_filename, "zip", output_path)
print(f"Model zipped to {zip_filename}.zip")

Model and tokenizer saved to ./pretrain_mdeberta
